

> Name: Saif Al-din Muhammad

> ID: 30808160102413-Eyouth

> Project: Project Library (2nd term project)







In [67]:
import sqlite3
import pandas as pd
import json
import numpy as np
import subprocess
import os

In [68]:
# Set your Student ID here
STUDENT_ID = "3080160102413"

In [94]:
!git config --global user.name "Seif Al-din Muhammad"
!git config --global user.email "Seif7reef@users.noreply.github.com"
!git init

Reinitialized existing Git repository in /content/.git/


In [92]:
# PART 1: ANSWERING THE 5 BUSINESS QUESTIONS
# ==========================================

conn = sqlite3.connect('library.db')

print("--- Question 1: How much is each member borrowing? ---")
q1_sql = """
SELECT
    m.member_id,
    m.first_name,
    m.last_name,
    COUNT(c.checkout_id) AS total_checkouts
FROM members m
LEFT JOIN checkouts c ON m.member_id = c.member_id
GROUP BY m.member_id, m.first_name, m.last_name;
"""
df_q1 = pd.read_sql_query(q1_sql, conn)
print(df_q1.head())


print("--- Question 2: Which books match a chosen author pattern? ---")
q2_sql = """
SELECT
    book_id,
    title,
    author
FROM books
WHERE author LIKE 'Karim%';
"""
df_q2 = pd.read_sql_query(q2_sql, conn)
print(df_q2)


print("--- Question 3: What are the most popular books? ---")
q3_sql = """
SELECT
    b.book_id,
    b.title,
    COUNT(c.checkout_id) AS checkout_count
FROM books b
JOIN checkouts c ON b.book_id = c.book_id
GROUP BY b.book_id, b.title
ORDER BY checkout_count DESC
LIMIT 5;
"""
df_q3 = pd.read_sql_query(q3_sql, conn)
print(df_q3)


print("--- Question 4: Who are the most active readers? ---")
q4_sql = """
SELECT
    m.member_id,
    m.first_name,
    m.last_name,
    COUNT(c.checkout_id) AS books_borrowed
FROM members m
JOIN checkouts c ON m.member_id = c.member_id
GROUP BY m.member_id, m.first_name, m.last_name
ORDER BY books_borrowed DESC
LIMIT 10;
"""
df_q4 = pd.read_sql_query(q4_sql, conn)
print(df_q4)


print("--- Question 5: Neighborhood activity further back in time ---")
q5_sql = """
SELECT
    c.checkout_id,
    m.member_id,
    m.first_name,
    m.last_name,
    m.neighborhood,
    c.checkout_date
FROM checkouts c
JOIN members m ON c.member_id = m.member_id
WHERE TRIM(LOWER(m.neighborhood)) = 'maadi'
ORDER BY c.checkout_date DESC
LIMIT 100 OFFSET 10;
"""
df_q5 = pd.read_sql_query(q5_sql, conn)
print(df_q5.head())

--- Question 1: How much is each member borrowing? ---
   member_id first_name last_name  total_checkouts
0       1001      Salma   Ibrahim                1
1       1002      Fares     Saleh                2
2       1003     Bassel    Hegazy                9
3       1004      Fares     Wahba                0
4       1005    Youssef     Halim                3
--- Question 2: Which books match a chosen author pattern? ---
   book_id                  title      author
0      521  A Garden of Equations  Karim Elwy
1      522    The Puzzle Merchant  Karim Elwy
--- Question 3: What are the most popular books? ---
   book_id                   title  checkout_count
0      501         The Silver Kite              57
1      507   Fossils and Fireflies              55
2      513  Circuits for Beginners              46
3      519        Kites Over Cairo              38
4      525    Storms and Sailboats              25
--- Question 4: Who are the most active readers? ---
   member_id first_name la

In [91]:
# PART 2: BRINGING THE THREE SOURCES TOGETHER
# ==========================================

conn = sqlite3.connect('library.db')

df_db_checkouts = pd.read_sql_query("SELECT * FROM checkouts", conn)
df_db_members = pd.read_sql_query("SELECT * FROM members", conn)
df_db_books = pd.read_sql_query("SELECT * FROM books", conn)

# Stage 1: Members and Checkouts
# ---------------------------------
stage_1 = pd.merge(df_db_checkouts, df_db_members, on='member_id', how='left')

# Stage 2: Book Details
# ------------------------
with open('books.json', 'r', encoding='utf-8') as f:
    json_books = json.load(f)
df_json_books = pd.DataFrame(json_books)

df_all_books = pd.merge(df_db_books, df_json_books, on='book_id', how='left')
stage_2 = pd.merge(stage_1, df_all_books, on='book_id', how='left')

# Stage 3: Summer Checkouts
# -------------------------
df_html_kickoff = pd.read_html('summer_checkouts.html')[0]
df_html_kickoff.columns = ['member_id', 'book_id', 'checkout_date']

df_html_merged = pd.merge(df_html_kickoff, df_db_members, on='member_id', how='left')
df_html_merged = pd.merge(df_html_merged, df_all_books, on='book_id', how='left')

final_task1_dataset = pd.concat([stage_2, df_html_merged], ignore_index=True)
conn.close()

In [90]:
# Save file
task1_filename = f"{STUDENT_ID}-combined_dataset_task1.csv"
final_task1_dataset.to_csv(task1_filename, index=False)

print("* TASK 1 COMPLETED *")
print(f"Successfully saved combined data to: {task1_filename}")

* TASK 1 COMPLETED *
Successfully saved combined data to: 3080160102413-combined_dataset_task1.csv


In [89]:
# Read and load combined file from Task 1
input_file = f"{STUDENT_ID}-combined_dataset_task1.csv"
df = pd.read_csv(input_file)

print(f"Loaded {input_file}. Initial shape:", df.shape)

Loaded 3080160102413-combined_dataset_task1.csv. Initial shape: (417, 17)


In [88]:
# Problem 1: Missing Values
# ----------------------------
if 'pages' in df.columns:
    df['pages'] = df['pages'].fillna(df['pages'].median())


# Problem 2: Removing Duplicates
# --------------------------------
initial_rows = len(df)
df = df.drop_duplicates()
print(f"Removed {initial_rows - len(df)} true duplicate rows.")

# Problem 3: The Same Value in Different ways
# ---------------------------------------------
if 'neighborhood' in df.columns:
    df['neighborhood'] = (
        df['neighborhood']
        .astype(str)
        .str.strip()
        .str.title()
        .str.replace(r'\s+', ' ', regex=True)
    )

if 'membership_status' in df.columns:
    df['membership_status'] = (
        df['membership_status']
        .astype(str)
        .str.strip()
        .str.capitalize()
    )

# Problem 4: Checkouts Checking
# ------------------------------
if 'first_name' in df.columns:
    unmatched_count = df['first_name'].isnull().sum()
    print(f"Found {unmatched_count} checkouts with unmatched member IDs.")
    df = df.dropna(subset=['first_name'])

Removed 0 true duplicate rows.
Found 0 checkouts with unmatched member IDs.


In [87]:
# Save Cleaned File
task2_filename = f"{STUDENT_ID}-task2_cleaned_data.csv"
df.to_csv(task2_filename, index=False)

print("* TASK 2 COMPLETED *")
print(f"Successfully saved cleaned data to: {task2_filename}")

* TASK 2 COMPLETED *
Successfully saved cleaned data to: 3080160102413-task2_cleaned_data.csv


In [76]:
# Load cleaned dataset from Task 2
df_clean = pd.read_csv(f"{STUDENT_ID}-task2_cleaned_data.csv")

In [86]:
print('Reloading df_clean to ensure it is defined...')
# Load cleaned dataset from Task 2
df_clean = pd.read_csv(f"{STUDENT_ID}-task2_cleaned_data.csv")
print(f"df_clean loaded with shape: {df_clean.shape}")

Reloading df_clean to ensure it is defined...
df_clean loaded with shape: (404, 17)


In [85]:
# PART 1: DATA FAIRNESS COMPARISON
# ===================================
print("=== Neighborhood Comparison (Members vs Checkouts) ===")
fairness_summary = df_clean.groupby('neighborhood').agg(
    total_members=('member_id', 'nunique'),
    total_checkouts=('checkout_id', 'count')
).reset_index()

# Calculate average checkouts per member in each neighborhood
fairness_summary['checkouts_per_member'] = (
    fairness_summary['total_checkouts'] / fairness_summary['total_members']
).round(2)

display(fairness_summary)

=== Neighborhood Comparison (Members vs Checkouts) ===


,neighborhood,total_members,total_checkouts,checkouts_per_member
0,Heliopolis,13,84,6.46
1,Maadi,20,106,5.30
2,Nasr City,16,97,6.06
3,Shubra,5,34,6.80
4,Zamalek,11,62,5.64


In [83]:
# PART 2: GIT VERSION CONTROL & LOG GENERATION
# ===============================================
os.system('git config --global user.name "Student"')
os.system('git config --global user.email "student@example.com"')
os.system('git init')

# Commit 1: Task 1 Initial Data Gathering
os.system(f'git add {STUDENT_ID}-combined_dataset_task1.csv')
os.system('git commit -m "Task 1: Completed data gathering and source merging"')

# Commit 2: Task 2 Data Cleaning & Integrity
os.system(f'git add {STUDENT_ID}-task2_cleaned_data.csv')
os.system('git commit -m "Task 2: Applied data cleaning and handled integrity issues"')

# Commit 3: Task 3 Analysis & Final Deliverables
os.system('git add .')
os.system('git commit -m "Task 3: Performed fairness evaluation and finalized submission"')

# Generate git_log.txt
git_log_output = subprocess.getoutput('git log --pretty=format:"Commit: %h | %s | %cd"')
with open(f"{STUDENT_ID}-git_log.txt", "w") as f:
    f.write(git_log_output)

In [93]:
!rm -rf .git
!git init

!git add 3080160102413-combined_dataset_task1.csv 3080160102413-task1_answers.txt
!git commit -m "Task 1: Add combined dataset and business query results"

!git add 3080160102413-task2_cleaned_data.csv 3080160102413-task2_integrity_report.txt
!git commit -m "Task 2: Add cleaned dataset and data integrity report"

!git add 3080160102413-task3_fairness_reflection.txt
!git commit -m "Task 3: Add fairness reflection and ethical guidelines"

!git log --pretty=format:"%h - %an, %ar : %s" > 3080160102413-git_log.txt
!git log --oneline

hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
Initialized empty Git repository in /content/.git/
[master (root-commit) 63a67cd] Task 1: Add combined dataset and business query results
 2 files changed, 466 insertions(+)
 create mode 100644 3080160102413-combined_dataset_task1.csv
 create mode 100644 3080160102413-task1_answers.txt
[master 5f75b25] Task 2: Add cleaned dataset and data integrity report
 2 files changed, 432 insertions(+)
 create mode 100644 3080160102413-task2_cleaned_data.csv
 create mode 100644 3080160102413-task2_integrity_report.txt
[ma